# PFP II companion: signed exchange and a finite search frontier

This executable companion to [Paper 002](../paper.md) checks supplied finite signed rewrite graphs and exact real matrices. It does not derive fermions, a Hamiltonian path integral, a physical sign-free representation, or a solution to the fermion sign problem. All code cells use Python's standard library and the two modules in `../code/genesis_alpha/`.

The three exercises distinguish a local sign equation, a checkable obstruction in a fixed transformation family, and an unfinished search. The scheduler below is deterministic threaded batch enumeration; **MCTS is not implemented**. Clear the kernel and run all cells in order.

In [1]:
import contextlib
import io
import sys
from dataclasses import replace
from hashlib import sha256
from pathlib import Path


def locate_finite_code():
    """Work from the package root or from its jupyter_notebooks folder."""
    for base in (Path.cwd(), Path.cwd().parent):
        candidate = base / "code" / "genesis_alpha"
        if (candidate / "sign_graph.py").is_file() and (candidate / "frontier_search.py").is_file():
            return candidate
    raise FileNotFoundError("Start the notebook from the package root or its jupyter_notebooks folder")


@contextlib.contextmanager
def one_output_block():
    """Send a cell's printed lines as one stream message.

    Timed flushes could otherwise split output into several messages, and a
    stored notebook would then differ from a fresh run under load.
    """
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            yield
    finally:
        sys.stdout.write(buffer.getvalue())
        sys.stdout.flush()


code_dir = locate_finite_code()
sys.path.insert(0, str(code_dir))
from sign_graph import (
    GaugeWitness, OddCycleWitness, SignedRewrite,
    classify_sign_graph, classify_matrix_sign_gauge,
    matrix_sign_graph, verify_gauge, verify_odd_cycle,
)
from frontier_search import (
    CHECKER_VERSION, DIAGONAL_Z2_GRAMMAR,
    PresentationKey, exact_matrix_digest, search_exact_matrix,
    verify_search_receipt,
)
with one_output_block():
    print("Loaded finite checkers from: code/genesis_alpha")

Loaded finite checkers from: code/genesis_alpha


## 1. An exchange edge is not yet a fermion obstruction

Supply a two-point carrier `left`, `right` and label its exchange edge odd. The equation for a vertex gauge is $\alpha(\mathrm{left})\oplus\alpha(\mathrm{right})=1$. It has a solution, so the free two-point swap alone has no sign holonomy. The odd character is **supplied input**, not a consequence of a bare primitive difference.

An *additional* odd self-loop demands $\alpha(\mathrm{left})\oplus\alpha(\mathrm{left})=1$, which is impossible. That loop is a chosen extension of the presentation. It must not be silently identified with the canonical two-point action groupoid or the Genesis certified quotient.

In [2]:
with one_output_block():
    vertices = ("left", "right")
    exchange = (SignedRewrite("left", "right", 1),)
    free_result = classify_sign_graph(vertices, exchange)
    assert isinstance(free_result, GaugeWitness)
    assert verify_gauge(vertices, exchange, free_result)
    print("Free swap: gauge bits", dict(zip(free_result.vertices, free_result.bits)))

    with_loop = exchange + (SignedRewrite("left", "left", 1),)
    loop_result = classify_sign_graph(vertices, with_loop)
    assert isinstance(loop_result, OddCycleWitness)
    assert verify_odd_cycle(vertices, with_loop, loop_result)
    print("Added odd self-loop: verified odd closed walk", loop_result.steps)

Free swap: gauge bits {'left': 0, 'right': 1}
Added odd self-loop: verified odd closed walk ((1, 'left', 'left'),)


## 2. A complete result for one fixed-basis matrix grammar

For an exact real symmetric matrix $H$, the adapter puts an edge between nonzero off-diagonal entries; its parity is `1` when $H_{ij}>0$. A diagonal sign choice $D_{ii}=(-1)^{\alpha_i}$ makes all off-diagonals of $DHD$ nonpositive exactly when every edge equation $\alpha_i\oplus\alpha_j=\mathrm{odd}_{ij}$ holds. A path of positive entries admits such a gauge. A positive triangle has odd cycle parity and does not admit one.

This is complete **only** for diagonal $\mathbb Z_2$ sign changes in the declared basis. The cycle does not rule out a different basis, path representation, regrouping, or a physical sign-free construction.

In [3]:
path_matrix = ((0, 1, 0),
               (1, 0, 1),
               (0, 1, 0))
triangle_matrix = ((0, 1, 1),
                   (1, 0, 1),
                   (1, 1, 0))

with one_output_block():
    path_vertices, path_edges = matrix_sign_graph(path_matrix)
    path_result = classify_matrix_sign_gauge(path_matrix)
    assert isinstance(path_result, GaugeWitness)
    assert verify_gauge(path_vertices, path_edges, path_result)
    print("Positive path: exact sign gauge", path_result.bits)

    triangle_vertices, triangle_edges = matrix_sign_graph(triangle_matrix)
    triangle_result = classify_matrix_sign_gauge(triangle_matrix)
    assert isinstance(triangle_result, OddCycleWitness)
    assert verify_odd_cycle(triangle_vertices, triangle_edges, triangle_result)
    print("Positive triangle: verified odd cycle", triangle_result.steps)

Positive path: exact sign gauge (0, 1, 0)
Positive triangle: verified odd cycle ((0, '1', '0'), (1, '0', '2'), (2, '2', '1'))


## 3. Budgeted proposal search keeps `UNKNOWN` until the frontier closes

A presentation key pins the exact matrix digest and the checker grammar. The other fields below describe a **toy matrix exercise**; the `adapter_digest` is a reproducible label for this exercise, **not** a proof that the matrix represents a physical fermion system. We spend three, then three, then two candidate checks on the triangle. The first two receipts leave an untouched suffix, so neither licenses a negative conclusion. The last receipt uses the independent odd-cycle checker after all eight vertex-bit assignments have been covered.

The implementation submits independent checks to a thread pool but consumes them in canonical order. It makes no multicore speedup claim.

In [4]:
key = PresentationKey(
    hamiltonian_digest=exact_matrix_digest(triangle_matrix),
    basis="declared-three-state-basis",
    ensemble="toy-none",
    filling="toy-not-defined",
    boundary="finite-three-state",
    representation="fixed-real-matrix",
    estimator="off-diagonal-sign-only",
    observables=("toy-matrix-sign",),
    beta="not-used",
    parameters=(),
    transformation_grammar=DIAGONAL_Z2_GRAMMAR,
    adapter_digest=sha256(b"paper-002-toy-matrix-adapter-label-v1").hexdigest(),
    checker_version=CHECKER_VERSION,
)
with one_output_block():
    first = search_exact_matrix(triangle_matrix, key, node_budget=3, workers=2)
    assert first.status == "UNKNOWN"
    assert verify_search_receipt(triangle_vertices, triangle_edges, key, first)
    print("First:", first.status, "next untested index", first.frontier.next_index,
          "receipt", first.receipt_sha256[:12])

    second = search_exact_matrix(
        triangle_matrix, key, node_budget=3, workers=2,
        frontier=first.frontier, prior_receipts=(first,),
    )
    assert second.status == "UNKNOWN"
    assert verify_search_receipt(triangle_vertices, triangle_edges, key, second, (first,))
    print("Second:", second.status, "next untested index", second.frontier.next_index,
          "receipt", second.receipt_sha256[:12])

    final = search_exact_matrix(
        triangle_matrix, key, node_budget=2, workers=2,
        frontier=second.frontier, prior_receipts=(first, second),
    )
    assert final.status == "RESTRICTED_OBSTRUCTION"
    assert isinstance(final.witness, OddCycleWitness)
    assert verify_search_receipt(triangle_vertices, triangle_edges, key, final, (first, second))
    print("Final:", final.status, "covered", final.frontier.next_index,
          "of", final.frontier.stop_index, "candidate gauges")
    print("Final odd cycle:", final.witness.steps)

First: UNKNOWN next untested index 3 receipt 7bd3521ede7c
Second: UNKNOWN next untested index 6 receipt 7d48a636cec5
Final: RESTRICTED_OBSTRUCTION covered 8 of 8 candidate gauges
Final odd cycle: ((0, '1', '0'), (1, '0', '2'), (2, '2', '1'))


### Receipt integrity and interpretation

The digest and replayed checks bind a receipt to this matrix's sign graph and key. A changed receipt is rejected; a later call also refuses to resume from it. This is a finite exact checker, not a proof of the Hamiltonian-to-graph adapter or observable preservation. A GIN-style index can retrieve proposals, but retrieved proposals have no authority until checked under the same full presentation key.

In [5]:
tampered_first = replace(first, receipt_sha256="0" * 64)
assert not verify_search_receipt(triangle_vertices, triangle_edges, key, tampered_first)
with one_output_block():
    try:
        search_exact_matrix(
            triangle_matrix, key, node_budget=3,
            frontier=first.frontier, prior_receipts=(tampered_first,),
        )
    except ValueError as exc:
        print("Tampered resume refused:", exc)
    else:
        raise AssertionError("A tampered receipt was accepted")

Tampered resume refused: prior receipts do not form a verified UNKNOWN chain
